# 02 — Data Cleaning & Preparation
**Objective 3:** Handle missing values, duplicates, clean review text, and build a reliable NLP corpus.


In [1]:
import re
from pathlib import Path

import pandas as pd
import numpy as np

PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data"
SENTIMENT_LABELS = ["negative", "neutral", "positive"]
MIN_REVIEW_LENGTH = 20

def load_df(stem: Path) -> pd.DataFrame:
    for s in (".parquet", ".pkl"):
        p = stem.with_suffix(s)
        if p.exists() and p.stat().st_size > 0:
            return pd.read_parquet(p) if s == ".parquet" else pd.read_pickle(p)
    raise FileNotFoundError(f"Run 01_data_collection first – missing {stem}")

df = load_df(DATA_DIR / "consolidated_reviews")
print("Loaded:", df.shape)
df.head(2)


Loaded: (35000, 24)


,review_id,booking_id,guest_id,hotel_id,review_text,rating_score,review_date,sentiment_label,bk_guest_id,bk_hotel_id,...,bk_total_amount,bk_stay_duration,gt_customer_type,gt_loyalty_status,gt_country,gt_previous_stays,ht_hotel_name,ht_location,ht_star_rating,ht_total_rooms
0,1,1,6699,18,The overall price was fair for what we paid. M...,3.2,2025-06-02,neutral,6699,18,...,296.44,1,Leisure,NaN,France,4,The Aberdeen Grand,Aberdeen,4,118
1,2,2,11380,23,The room was nothing out of the ordinary for w...,4.0,2025-01-22,positive,11380,23,...,483.56,4,Business,Gold,Germany,4,Nottingham Park Hotel,Nottingham,4,95


In [2]:
# Text cleaning helpers
_URL_RE = re.compile(r"https?://\S+|www\.\S+")
_EMAIL_RE = re.compile(r"\S+@\S+")
_HTML_RE = re.compile(r"<[^>]+>")
_MULTI_SPACE_RE = re.compile(r"\s+")

def clean_review_text(text: str) -> str:
    """Light cleaning for transformers (keeps case)."""
    if not isinstance(text, str) or not text.strip():
        return ""
    text = _HTML_RE.sub(" ", text)
    text = _URL_RE.sub(" ", text)
    text = _EMAIL_RE.sub(" ", text)
    text = re.sub(r"[^\w\s'.,!?-]", " ", text)
    text = _MULTI_SPACE_RE.sub(" ", text).strip()
    return text

def clean_for_classical_ml(text: str) -> str:
    """Aggressive cleaning for TF-IDF baselines."""
    text = clean_review_text(text).lower()
    text = re.sub(r"[^a-z\s']", " ", text)
    return _MULTI_SPACE_RE.sub(" ", text).strip()

# Demo
sample = df["review_text"].iloc[0]
print("RAW :", sample)
print("CLEAN:", clean_review_text(sample))
print("ML  :", clean_for_classical_ml(sample))


RAW : The overall price was fair for what we paid. Might consider it again, might not.
CLEAN: The overall price was fair for what we paid. Might consider it again, might not.
ML  : the overall price was fair for what we paid might consider it again might not


In [3]:
# Full cleaning pipeline
out = df.copy()

# 1. Deduplicate
n0 = len(out)
out = out.drop_duplicates(subset=["review_id"], keep="first")
print(f"Dropped {n0 - len(out)} duplicate review_ids")

# 2. Empty text
out = out[out["review_text"].notna() & (out["review_text"].str.strip() != "")]
print(f"After empty-text filter: {len(out)}")

# 3. Clean text
out["review_text_clean"] = out["review_text"].apply(clean_review_text)
out["review_text_ml"] = out["review_text"].apply(clean_for_classical_ml)

# 4. Min length
out["text_length"] = out["review_text_clean"].str.len()
out = out[out["text_length"] >= MIN_REVIEW_LENGTH]
print(f"After min-length ({MIN_REVIEW_LENGTH}): {len(out)}")

# 5. Validate labels & ratings
out["sentiment_label"] = out["sentiment_label"].str.lower().str.strip()
out = out[out["sentiment_label"].isin(SENTIMENT_LABELS)]
out["rating_score"] = pd.to_numeric(out["rating_score"], errors="coerce")
out = out[out["rating_score"].between(1.0, 5.0)]

# 6. Loyalty NaN → None
if "gt_loyalty_status" in out.columns:
    out["gt_loyalty_status"] = out["gt_loyalty_status"].fillna("None")

# 7. Word count
out["word_count"] = out["review_text_clean"].str.split().str.len()

cleaned = out.reset_index(drop=True)
print("Cleaned shape:", cleaned.shape)
cleaned[["review_id", "sentiment_label", "rating_score", "text_length", "word_count"]].head()


Dropped 0 duplicate review_ids
After empty-text filter: 35000
After min-length (20): 35000
Cleaned shape: (35000, 28)


,review_id,sentiment_label,rating_score,text_length,word_count
0,1,neutral,3.2,80,15
1,2,positive,4.0,158,28
2,3,negative,1.9,100,15
3,4,negative,2.2,129,21
4,5,neutral,3.3,196,39


In [4]:
# Data quality checks
checks = {
    "n_rows": len(cleaned),
    "n_unique_reviews": cleaned["review_id"].nunique(),
    "null_review_text": int(cleaned["review_text_clean"].isna().sum()),
    "empty_review_text": int((cleaned["review_text_clean"].str.strip() == "").sum()),
    "invalid_sentiment": int((~cleaned["sentiment_label"].isin(SENTIMENT_LABELS)).sum()),
    "rating_out_of_range": int((~cleaned["rating_score"].between(1, 5)).sum()),
    "duplicate_review_ids": int(cleaned["review_id"].duplicated().sum()),
    "min_text_length": int(cleaned["text_length"].min()),
    "max_text_length": int(cleaned["text_length"].max()),
    "mean_text_length": float(cleaned["text_length"].mean()),
}
for k, v in checks.items():
    print(f"  {k}: {v}")


  n_rows: 35000
  n_unique_reviews: 35000
  null_review_text: 0
  empty_review_text: 0
  invalid_sentiment: 0
  rating_out_of_range: 0
  duplicate_review_ids: 0
  min_text_length: 67
  max_text_length: 278
  mean_text_length: 171.02994285714286


In [5]:
# Save cleaned corpus
path = DATA_DIR / "cleaned_reviews.parquet"
try:
    cleaned.to_parquet(path, index=False)
except Exception:
    path = DATA_DIR / "cleaned_reviews.pkl"
    cleaned.to_pickle(path)
print(f"Saved → {path} ({len(cleaned):,} rows)")


Saved → /Users/amadionyedika/Desktop/AMDARI INTERNSHIP/guest_experience_analytics 2/data/cleaned_reviews.parquet (35,000 rows)
